# 📰 News Topic Classifier Using BERT

## Problem Statement
Classify news articles into different topic categories using a pre-trained BERT transformer model.

## Objective
- Fine-tune BERT model on AG News dataset
- Classify news into 4 categories: World, Sports, Business, Sci/Tech
- Evaluate model accuracy and F1-score
- Deploy as an interactive application

## Dataset
- **AG News Dataset** - 120,000 news articles
- **Classes**: World, Sports, Business, Science/Technology
- **Features**: Article Title + Description

In [1]:
!pip install pandas
!pip install evaluate accelerate
!pip install numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [3]:
import kagglehub
import os

print("Downloading AG News dataset...")
download_path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
file_path = os.path.join(download_path, "train.csv")

df = pd.read_csv(
    file_path, 
    names=["Class Index", "Title", "Description"], 
    encoding="utf-8",
    engine="python",
    on_bad_lines="warn"
)

df = df.sample(n=20000, random_state=42)

print(f" Loaded {len(df)} articles\n")

print("Dataset Info:")
print(f"  - Shape: {df.shape}")
print(f"  - Columns: {df.columns.tolist()}")
print("\nFirst 3 rows:")
print(df.head(3))

C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Loaded 20000 articles

Dataset Info:
  - Shape: (20000, 3)
  - Columns: ['Class Index', 'Title', 'Description']

First 3 rows:
      Class Index                                              Title  \
71788           3       BBC set for major shake-up, claims newspaper   
67218           4                       Taking Microsoft for a spin?   
54066           3  September sales at Target stores beat retail a...   

                                             Description  
71788  London - The British Broadcasting Corporation,...  
67218  The software juggernaut that conquered the des...  
54066  MINNEAPOLIS - While other retailers struggled ...  


In [4]:
from transformers import AutoTokenizer

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [5]:
import pandas as pd
from datasets import Dataset

df['Class Index'] = pd.to_numeric(df['Class Index'], errors='coerce')

df = df.dropna(subset=['Class Index'])

df['labels'] = df['Class Index'].astype(int) - 1

df['text'] = df['Title'].astype(str) + " " + df['Description'].astype(str)

hf_dataset = Dataset.from_pandas(df[['text', 'labels']])

def tokenize_batch(batch):
    return tokenizer(
        batch["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

dataset_tokenized = hf_dataset.map(tokenize_batch, batched=True)
dataset_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Cleaned! Remaining rows: {len(df)}")

Map: 100%|██████████████████████████████████████████████████████████████| 19999/19999 [00:04<00:00, 4431.18 examples/s]

Cleaned! Remaining rows: 19999


In [6]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np





In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", 
    num_labels=4
)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2692.13it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

In [8]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    
    return {**acc, **f1}

In [9]:

training_args = TrainingArguments(
    output_dir="./bert_news_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    TENSORBOARD_LOGGING_DIR='./logs',
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenized, # Using your tokenized dataset from Task 1
    eval_dataset=dataset_tokenized.select(range(500)), # Small subset for quick eval
    compute_metrics=compute_metrics,
)


In [11]:


trainer.train()

C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.261433,0.189858,0.940000,0.940232
2,0.149864,0.119184,0.972000,0.971950
3,0.100575,0.094013,0.982000,0.981947


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.56it/s]
C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.57it/s]
C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.31it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.Lay

TrainOutput(global_step=3750, training_loss=0.1918805160522461, metrics={'train_runtime': 20711.3072, 'train_samples_per_second': 2.897, 'train_steps_per_second': 0.181, 'total_flos': 3946539364604928.0, 'train_loss': 0.1918805160522461, 'epoch': 3.0})

In [13]:
model_path = "./final_bert_news_model"

trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved successfully to {model_path}!")

Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.18it/s]

Model saved successfully to ./final_bert_news_model!


In [14]:
eval_results = trainer.evaluate()

print(f"--- Evaluation Results ---")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1 Score:  {eval_results['eval_f1']:.4f}")
print(f"Loss:      {eval_results['eval_loss']:.4f}")

C:\Users\aliim\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.100575,0.094013,3,0.982000,0.981947


--- Evaluation Results ---
Accuracy: 0.9820
F1 Score:  0.9819
Loss:      0.0940


In [15]:
output_dir = "./saved_bert_news_model"

model.save_pretrained(output_dir)

tokenizer.save_pretrained(output_dir)

print(f"Project state saved to {output_dir}")

Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.13it/s]

Project state saved to ./saved_bert_news_model
